## Webots Analysis

In [ ]:
import torch
import numpy as np
import os

from utils import build_dataloader, WebotsFrameDataset, set_seed
from attention_model.conv_encoder import ConvEncoder

from torchvision.transforms import v2
import matplotlib.pyplot as plt

from sklearn.cluster import OPTICS, KMeans, BisectingKMeans, SpectralClustering
from typing import Optional


print(f'PyTorch version: {torch.__version__}')
print('Numpy version:', np.__version__)

seed = 42
set_seed(seed)


feature_model_path = './attention_model/SAM_weights/'

# AE model
pooling = (2, 2)
hidden_dims = (512, )
n_bottleneck_layer = 200
checkpoint_path = './ae_model/feature_extractor_ae_checkpoint/features_only_pool2x2/'

# images must be in the SAME dir as the csv file
webots_dataset_path = '../Denis/HIP_AE_VISUAL/Datasets/Tmaze_2/data.csv'

do_training = False

figs_dir = './figs/'
if figs_dir:
    os.makedirs(figs_dir, exist_ok=True)

In [ ]:
batch_size = 512
tf = v2.Compose([
    v2.ToDtype(torch.float32, scale=True),
    v2.Resize((248, 328)),
    v2.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])


loader = build_dataloader(
    data_csv=webots_dataset_path, 
    transform=tf, 
    batch_size=batch_size, 
    shuffle=True,
    seed=seed
)

reps = 1
for _ in range(reps):
    images, xy = next(iter(loader))

print(images.shape, images.dtype)
print(xy.shape, xy.dtype)

# show one image
img = images[0].permute(1, 2, 0).numpy()
img = (img * [0.229, 0.224, 0.225]) + [0.485, 0.456, 0.406]  # unnormalize
img = (img * 255).astype('uint8')
plt.imshow(img)
plt.show()


In [ ]:
# --- Load model ---
ckpt = torch.load(feature_model_path + "best.ckpt", weights_only=False, map_location=torch.device('cpu'))

encoder_state_dict = ckpt["state_dict"]
# Fix: only delete the OLD key if it differs from the new one
for k in list(encoder_state_dict.keys()):
    new_k = k.replace("encoder.encoder", "encoder").replace("encoder.concept_proj", "concept_proj")
    if new_k != k:
        encoder_state_dict[new_k] = encoder_state_dict.pop(k)

feature_extractor = ConvEncoder()
feature_extractor.load_state_dict(ckpt["state_dict"])
feature_extractor.eval()

# --- Load PFC templates ---
pfc_templates = torch.load(feature_model_path + "pfc-templates.pth", weights_only=False).numpy()


# Use first image
img_tf = images[0:1].float()  # (1, C, H, W)

# --- Extract features ---
with torch.inference_mode():
    features = feature_extractor(img_tf).squeeze(0).numpy()  # (C, H, W)

print("Feature map shape:", features.shape)
print("min/max values:", features.min(), features.max())

n_channels = features.shape[0]
features_hwc = np.moveaxis(features, 0, -1)  # (H, W, C)

# --- All feature maps ---
ncols = 32
nrows = int(np.ceil(n_channels / ncols))

fig, axs = plt.subplots(nrows, ncols, figsize=(ncols * 0.8, nrows * 0.8))
axs = axs.flat
for i in range(n_channels):
    axs[i].imshow(features_hwc[:, :, i], cmap='inferno', vmin=features.min(), vmax=features.max())
    axs[i].axis('off')
# Hide unused axes
for j in range(n_channels, nrows * ncols):
    axs[j].set_visible(False)

plt.subplots_adjust(wspace=0.05, hspace=0.05)
plt.show()


In [ ]:
h, w, c = features_hwc.shape
feature_reshaped = np.reshape(features_hwc, (h*w,c))

n_labels = [4, 6, 8, ]

for k in n_labels:
        clustering = OPTICS(min_samples=20).fit(feature_reshaped)
        kmeans = KMeans(n_clusters=k,random_state=0, n_init=100).fit(feature_reshaped)
        bisect_means = BisectingKMeans(n_clusters=k, random_state=0,n_init=5).fit(feature_reshaped)
        spectral_clustering = SpectralClustering(n_clusters=k,
                assign_labels='cluster_qr',
                random_state=0).fit(feature_reshaped)


        labels_bisec = bisect_means.labels_
        labels_kmean = kmeans.labels_
        labels_optics = clustering.labels_
        labels_spectral = spectral_clustering.labels_

        bisec_img = np.reshape(labels_bisec, (h,w))
        kmean_img = np.reshape(labels_kmean, (h,w))
        optics_img = np.reshape(labels_optics, (h,w))
        spectral_img = np.reshape(labels_spectral, (h,w))

        fig,axs = plt.subplots(1,4, figsize=(16,4))
        axs[0].imshow(bisec_img, cmap='inferno')
        axs[0].set_title('KMeans-Bisec')
        axs[1].imshow(kmean_img, cmap='inferno')
        axs[1].set_title('KMeans')
        axs[2].imshow(optics_img, cmap='inferno')
        axs[2].set_title('Optics')
        axs[3].imshow(spectral_img, cmap='inferno')
        axs[3].set_title('Spectral')
        for ax in axs:
                ax.axis('off')
        plt.suptitle(f'Clustering with k={k}')
        plt.show()

## Autoencoder integration


In [ ]:
from ae_model.dense_hippocampal_ae import PooledDenseAE
from utils import timer
from torch import optim, nn

from training_functions import train
from utils import get_parameters


# params
learning_rate = 5e-4
min_learning_rate = 1e-6
alpha = 1e5
C_factor = 1000.0
num_epochs = 4

# training loop

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if do_training:
    ae_model = PooledDenseAE(
        n_hidden=n_bottleneck_layer, 
        hidden_dims=hidden_dims,
        last_layer_activation=nn.Sigmoid(), 
        pool_output_size=pooling, 
        d_aux=None
    )
    print(f"Number of parameters: {get_parameters(ae_model)} M")

    optimizer = optim.Adam(ae_model.parameters(), lr=learning_rate)
    criterion = nn.MSELoss()
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=min_learning_rate)

    with timer("Training complete in: "):
        history = train(
            ae_model=ae_model, 
            feature_extractor=feature_extractor, 
            num_epochs=num_epochs,
            loader=loader, 
            optimizer=optimizer, 
            criterion=criterion, 
            alpha=alpha, 
            C_factor=C_factor,
            device=device,
            checkpoint_path=None,
            patience=5,
        )
        plt.plot(history)
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.show()
else:
    from main import load_run_config, build_model_from_config
    ae_model, _, _ = build_model_from_config(checkpoint_dir=checkpoint_path, device=device)

In [ ]:
from training_functions import get_eval_metrics

with timer("Extracting embeddings in: "):
    embeddings, positions, r2_per_batch = get_eval_metrics(
        dataloader=loader, 
        ae_model=ae_model, 
        feature_extractor=feature_extractor, 
        device=device, 
        feature_reconstruction_path=os.path.join(checkpoint_path, "reconstructed_features/"),  # save reconstructed features for visualization set to None if you don't want to save the reconstructed features
        prob_plot = 0.1,
)

labels = np.argmax(embeddings, axis=1)

In [ ]:
print("Accuracies:", np.mean(r2_per_batch))

In [ ]:
from ae_model.plotting import ratemaps, stats_place_fields, plot_ratemaps

all_ratemaps = ratemaps(embeddings, positions, n_bins=50, filter_width=3)

# Ratemap shape should be 200 x 50 x 50 (number of units in the latent space x numbe of bins x number of bins)
print(f'Ratemap shape: {all_ratemaps.shape}')

In [ ]:
plot_ratemaps(all_ratemaps, '', save=False)

In [ ]:
ratemap_sum = all_ratemaps.sum(axis=0)

fig, ax = plt.subplots()
fig.set_size_inches(3, 3)

plt.imshow(ratemap_sum, cmap='hot', origin='lower')

plt.show()

In [ ]:
sscp = np.dot(embeddings.T, embeddings)

fig = plt.figure(figsize=(7,6))
plt.title('Sums of Squares and Cross Products Matrix', fontsize=15)
plt.imshow(sscp, origin='lower')
plt.xlabel('Units')
plt.ylabel('Units')
plt.colorbar()
plt.show()

In [ ]:
from ae_model.plotting import format_centroids, plot_single_ratemap_density

# Define fields, centroids and sizes
peak_as_centroid=True
min_pix_cluster=0.02 #0.02
max_pix_cluster=0.5 #0.5
active_threshold=0.2 #0.2

all_num_fields, centroids, sizes = stats_place_fields(all_ratemaps, peak_as_centroid=peak_as_centroid, min_pix_cluster=min_pix_cluster, max_pix_cluster=max_pix_cluster, active_threshold=active_threshold)
centroids_per_field, sizes_per_field = format_centroids(all_num_fields, centroids, sizes)

In [ ]:
cell_id = np.random.randint(all_ratemaps.shape[0])
print(f'Cell Id: {cell_id}')
unit = np.random.randint(cell_id)
plot_single_ratemap_density(all_ratemaps, unit, all_num_fields, sizes_per_field, centroids_per_field, '', figsize=(5,5), save=False)

In [ ]:
heatmap, xedges, yedges = np.histogram2d(centroids[:,1], centroids[:,0], bins=(20, 20))
extent = [xedges[0], xedges[-1], yedges[0], yedges[-1]]

fig = plt.figure(figsize=(7,6))
plt.imshow(heatmap.T, extent=extent, origin='lower', cmap='viridis')
plt.colorbar(label='Density')
plt.title('Centroids density', fontsize=15)
plt.show()

In [ ]:
fig = plt.figure(figsize=(7,7))
for i in range(len(centroids)):
    plt.plot(centroids[:,1], centroids[:,0], 'bx', markersize=3)
plt.title("Centroids (n = " + str(len(centroids)) + ")", fontsize=15)
plt.show()

In [ ]:
counts = dict(zip(*np.unique(all_num_fields, return_counts=True)))
n_units = all_ratemaps.shape[0]
categories = ['0', '1', '2', '3']
values = [100 * counts.get(i, 0) / n_units for i in range(4)]

fig = plt.figure(figsize=(7,6))
plt.bar(categories, values)
plt.title('% of place fields', fontsize=15)
plt.xlabel('Place fields')
plt.ylabel('% of units')
plt.show()

In [ ]:
plt.figure(figsize=(6, 6))

plt.subplot(111)
plt.scatter(positions[:, 0], positions[:, 1], c=labels, cmap='rainbow', s=1, alpha=0.8)

# plt.scatter(prototypes[:, 0], prototypes[:, 1], marker='X', s=200, edgecolors='k', facecolors='none', linewidths=1.5)
plt.title("Tiling of the 2‑D space by place cells")
plt.axis('equal')


plt.tight_layout()
plt.show()